# Projeto Fictus | Análise Financeira — Bloco 4: Capital e Escalabilidade

---

## Pergunta Central do Bloco
> **A estrutura financeira da empresa-alvo é compatível com crescimento — e o modelo de recebimento escala junto com o volume operacional?**

---

## Contexto do Bloco

Os blocos anteriores estabeleceram o diagnóstico (Bloco 1), o perfil de risco (Bloco 2) e a rentabilidade esperada (Bloco 3). Este bloco fecha a análise financeira respondendo a dimensão de escala e capital: **a estrutura de recebimento da empresa-alvo exige capital crescente conforme o volume aumenta — e esse comprometimento pode criar um teto de crescimento que o comprador precisa conhecer antes de fechar o negócio.**

Uma empresa pode crescer em receita e, simultaneamente, comprimir sua posição de caixa — se o modelo de crédito não escalar proporcionalmente. Este é o ponto de maior assimetria da análise: os riscos de capital podem se materializar de forma abrupta, enquanto os benefícios do crescimento chegam de forma gradual.

**Este bloco investiga:**
1. Quanto capital está imobilizado na estrutura atual e como esse número evolui com o crescimento?
2. O crescimento de vendas consome ou gera caixa e em que ponto a operação precisa de funding externo?
3. O modelo financeiro escala junto com o varejo ou há um teto de crescimento que o comprador herdaria?
4. Quais são os vetores de ruptura prioritários e quais os gatilhos que o comprador precisará monitorar?

---


## Configuração e Carregamento

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
from pathlib import Path

try:
    _base = Path(__file__).resolve().parent
except NameError:
    _base = Path().resolve()
def _find_base(start: Path) -> Path:
    for p in [start, start.parent, start.parent.parent]:
        if (p / "data").exists() or (p / "notebooks").exists():
            return p
    return start
BASE_DIR = _find_base(_base)
DIR_FIN     = BASE_DIR / "data" / "finance"
DIR_EXPORTS = BASE_DIR / "exports"
DIR_EXPORTS.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore")

COR_RECEITA  = "#1B4F72"
COR_MARGEM   = "#27AE60"
COR_ALERTA   = "#C0392B"
COR_NEUTRO   = "#7F8C8D"
COR_DESTAQUE = "#E67E22"
COR_ROXO     = "#8E44AD"

sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 150, "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "axes.spines.top": False, "axes.spines.right": False,
})
def fmt_brl(x, pos=None):
    if abs(x) >= 1_000_000: return f"R$ {x/1_000_000:.1f}M"
    if abs(x) >= 1_000:     return f"R$ {x/1_000:.0f}K"
    return f"R$ {x:.0f}"
def fmt_pct(x, pos=None): return f"{x:.1f}%"
def salvar(fig, nome):
    caminho = DIR_EXPORTS / f"{nome}.png"
    fig.savefig(caminho)
    print(f"  → Salvo: {caminho.name}")

def ler(f, **kw):
    df = pd.read_csv(DIR_FIN / f, low_memory=False, **kw)
    df.columns = df.columns.str.strip()
    return df

fin_fato   = ler("fin_fato.csv")
fin_mensal = ler("fin_mensal.csv")
fin_trim   = ler("fin_trimestral.csv")
fin_pag    = ler("fin_pagamento.csv")

for col in ["preco", "numero_parcelas", "pmr_ajustado", "spread_intermediario",
            "capital_em_aberto", "anomalia_pagamento"]:
    if col in fin_fato.columns:
        fin_fato[col] = pd.to_numeric(fin_fato[col], errors="coerce")

# ─── Premissas centrais — todas auditáveis ────────────────────────────────────
CUSTO_OPERACIONAL_MENSAL = 25_000   # R$/mês
CUSTO_CAPITAL_AM         = 0.0120   # 1,2% a.m.
TAXA_INADIMPLENCIA       = fin_fato["anomalia_pagamento"].mean()
CUSTO_SETUP              = 150_000  # R$ — investimento único
CAPACIDADE_MAX_CARTEIRA  = 2_000_000  # R$ — limite máximo de carteira própria estimado

receita_total    = fin_fato["preco"].sum()
capital_total    = fin_fato["capital_em_aberto"].sum()
spread_total     = fin_fato["spread_intermediario"].sum()
meses_total      = fin_mensal["ano_mes"].nunique()
receita_mensal   = receita_total / meses_total
capital_mensal   = capital_total / meses_total

print(f"Dados: {len(fin_fato):,} registros | {meses_total} meses")
print(f"Receita mensal média  : R$ {receita_mensal:,.0f}")
print(f"Capital aberto médio  : R$ {capital_mensal:,.0f}")
print(f"Taxa de inadimplência : {TAXA_INADIMPLENCIA*100:.2f}%")


---

## Análise 1 — Quanto capital está imobilizado na estrutura atual e como esse número evolui com o crescimento?

> *"O modelo de vendas parceladas cria uma necessidade estrutural de capital imobilizado na carteira de recebíveis. Esta análise quantifica o volume de recursos comprometidos em diferentes cenários de crescimento e confronta essa necessidade com o teto de capacidade declarado nas premissas. O cruzamento entre demanda de capital e teto disponível define o fôlego financeiro do ativo — o limite a partir do qual o crescimento projetado exigiria funding externo não previsto na estrutura atual."*

**Framework:** Análise de Capital Imobilizado + Análise de Payback  
**Entrega:** Modelagem do capital em aberto necessário por cenário de volume e prazo

**Como este script responde à pergunta:**
> O script projeta o capital necessário na carteira para cinco cenários de volume e confronta com a capacidade máxima estimada. Dois painéis respondem à pergunta:
> 1. **Capital necessário por cenário de volume (barras):** Cinco barras para ×1,0, ×1,2, ×1,5, ×2,0 e ×3,0 o volume atual — coloridas em verde quando abaixo da capacidade máxima, laranja quando próximas e vermelho quando ultrapassam. A linha tracejada roxa é o teto: barras que tocam ou cruzam essa linha sinalizam o nível a partir do qual o comprador precisaria de funding externo para sustentar o crescimento. Cada barra é anotada com o valor em R$.
> 2. **Capital em aberto mensal observado:** Linha com área sombreada mostrando a evolução real do capital em aberto mês a mês, com a mesma linha de capacidade máxima sobreposta. Permite comparar o nível atual com o teto e visualizar se a operação já está se aproximando do limite mesmo sem crescimento adicional.

**Análise do Resultado:**
O gráfico de cenários revela se o ativo já opera próximo do seu teto atual — mesmo sem crescimento adicional. Se a barra do cenário ×1,0 já se aproxima da linha de capacidade máxima, o comprador herda uma operação com margem de escala reduzida desde o dia um. Para os cenários de expansão, cada barra que cruza o teto representa um nível de crescimento que exigirá funding externo ou revisão da estrutura de capital — e o valor do aporte necessário pode ser lido diretamente da distância entre a barra e a linha. Essa leitura transforma a ambição de crescimento em uma necessidade de capital concreta e auditável.


In [ ]:
# ─── Modelagem do capital imobilizado ────────────────────────────────────────
fatores_crescimento = [1.0, 1.2, 1.5, 2.0, 3.0]
capital_por_cenario = [
    capital_mensal * f for f in fatores_crescimento
]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Bloco 4 — Capital Imobilizado e Comprometimento por Cenário",
             fontsize=13, fontweight="bold")

# Painel 1: Capital necessário por cenário de crescimento
cores_cap = [COR_MARGEM, COR_MARGEM, COR_DESTAQUE, COR_ALERTA, COR_ALERTA]
bars = axes[0].bar([f"×{f:.1f} volume" for f in fatores_crescimento],
                   [c / 1000 for c in capital_por_cenario],
                   color=cores_cap, alpha=0.85)
axes[0].axhline(CAPACIDADE_MAX_CARTEIRA / 1000, color=COR_ROXO, linestyle="--",
                linewidth=1.5, label=f"Capacidade máx estimada: R${CAPACIDADE_MAX_CARTEIRA/1000:.0f}K")
axes[0].set_title("Capital Necessário na Carteira por Cenário de Volume (R$ mil)", fontsize=10)
axes[0].set_ylabel("Capital (R$ mil)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R${x:,.0f}K"))
axes[0].legend(fontsize=8)
for bar, val in zip(bars, capital_por_cenario):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f"R${val/1000:,.0f}K", ha="center", va="bottom", fontsize=9)

# Painel 2: Evolução do capital em aberto mensal real
axes[1].fill_between(range(len(fin_mensal)),
                     fin_mensal["capital_aberto"].fillna(0) / 1000,
                     alpha=0.3, color=COR_RECEITA)
axes[1].plot(range(len(fin_mensal)),
             fin_mensal["capital_aberto"].fillna(0) / 1000,
             color=COR_RECEITA, linewidth=2)
axes[1].axhline(CAPACIDADE_MAX_CARTEIRA / 1000, color=COR_ROXO, linestyle="--",
                linewidth=1.5, alpha=0.7, label="Cap. máx estimada")
axes[1].set_xticks(range(len(fin_mensal)))
axes[1].set_xticklabels(fin_mensal["ano_mes"], rotation=45, fontsize=7)
axes[1].set_title("Capital em Aberto Mensal Observado (R$ mil)", fontsize=10)
axes[1].set_ylabel("Capital (R$ mil)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R${x:,.0f}K"))
axes[1].legend(fontsize=8)

plt.tight_layout()
salvar(fig, "04_capital_imobilizado")
plt.show()

print("\n── Capital por Cenário ──────────────────────")
for f, c in zip(fatores_crescimento, capital_por_cenario):
    status = "✅" if c < CAPACIDADE_MAX_CARTEIRA else "⚠️ "
    print(f"  Volume ×{f:.1f}: capital necessário R$ {c:,.0f} {status}")


---

## Análise 2 — O crescimento de vendas consome ou gera caixa e em que ponto a operação precisa de funding externo?

> *"Diferente do lucro contábil, o fluxo de caixa acumulado revela a liquidez real da operação. Esta projeção de 12 meses simula o impacto do crescimento acelerado no caixa da empresa-alvo. O objetivo é mapear o "vale do fluxo de caixa" — o período inicial onde a expansão de parcelamentos consome mais recursos do que os recebimentos que entram — e determinar se o modelo atual é autofinanciável ou se exige aporte de capital para crescer."*

**Framework:** Análise de fluxo de caixa — Controle de processo  
**Entrega:** Ciclo de caixa do modelo atual por cenário de crescimento

**Como este script responde à pergunta:**
> O script projeta o fluxo de caixa acumulado por 12 meses para três cenários de crescimento e calcula a necessidade de funding por fator de volume. Dois painéis respondem à pergunta:
> 1. **Fluxo de caixa acumulado — 12 meses (três cenários):** Três linhas — verde para sem crescimento, azul para crescimento moderado e vermelho para crescimento agressivo — partindo do investimento inicial negativo e subindo pelo spread líquido mensal. O cruzamento de cada linha com o zero é o payback por cenário. Linhas que não cruzam o zero em 12 meses sinalizam risco de liquidez estrutural — o comprador precisaria de capital adicional para sustentar aquele ritmo de crescimento.
> 2. **Necessidade de funding adicional por fator de volume:** Barras que mostram quanto capital externo seria necessário para sustentar cada nível de crescimento além da capacidade própria. Barras verdes indicam que o modelo é auto-financiado; barras vermelhas indicam o valor de aporte necessário em R$.

**Análise do Resultado:**
Linhas de fluxo acumulado que demoram a cruzar o eixo zero indicam um modelo de alta intensidade de capital — o spread gerado mensalmente é insuficiente para recuperar o investimento inicial em ritmo acelerado. Se o cenário de crescimento agressivo apresentar saldo acumulado persistentemente negativo ao longo dos 12 meses projetados, o ativo possui risco de liquidez estrutural que precisa ser precificado no valuation. As barras de funding adicional traduzem esse risco em valor concreto: o aporte necessário para cada nível de crescimento que ultrapassa a capacidade de autofinanciamento do modelo atual.


In [ ]:
# ─── Ciclo de caixa: projeção de 12 meses ─────────────────────────────────────
# Recálculo defensivo: garante que as variáveis existam se a célula for rodada isolada
try:
    _ = spread_total
except NameError:
    spread_total  = fin_fato["spread_intermediario"].sum()
    meses_total   = fin_mensal["ano_mes"].nunique()
    receita_mensal = fin_fato["preco"].sum() / meses_total
    capital_mensal = fin_mensal["capital_aberto"].fillna(0).mean()

spread_liq_mensal_base = (
    (spread_total / meses_total)
    - receita_mensal * TAXA_INADIMPLENCIA
    - capital_mensal * CUSTO_CAPITAL_AM
    - CUSTO_OPERACIONAL_MENSAL
)

fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle("Bloco 4 — Ciclo de Caixa e Necessidade de Funding por Crescimento",
             fontsize=13, fontweight="bold")

# Painel 1: Fluxo de caixa acumulado por cenário (12 meses)
horizontes = list(range(13))
cenarios_caixa = {
    "Sem crescimento (×1.0)"     : 1.0,
    "Crescimento moderado (×1.2)" : 1.2,
    "Crescimento agressivo (×1.5)": 1.5,
}
cores_cenarios = [COR_MARGEM, COR_RECEITA, COR_ALERTA]

for (label, fator), cor in zip(cenarios_caixa.items(), cores_cenarios):
    delta_capital  = capital_mensal * (fator - 1.0)
    spread_m       = spread_liq_mensal_base * fator
    fluxo_acum     = []
    saldo          = -CUSTO_SETUP - delta_capital
    for m in horizontes:
        if m > 0:
            saldo += spread_m
        fluxo_acum.append(saldo / 1000)
    axes[0].plot(horizontes, fluxo_acum, color=cor, linewidth=2,
                 marker="o", markersize=4, label=label)

axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("Fluxo de Caixa Acumulado — Modelo Internalizado (R$ mil, 12 meses)", fontsize=10)
axes[0].set_xlabel("Meses após implantação")
axes[0].set_ylabel("Saldo Acumulado (R$ mil)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R${x:,.0f}K"))
axes[0].legend(fontsize=8)

# Painel 2: Necessidade de funding por cenário de volume
fatores_vol    = [1.0, 1.2, 1.5, 2.0, 3.0]
rotulos_vol    = [f"×{f:.1f}" for f in fatores_vol]
funding_necessario = [
    max(0.0, capital_mensal * f - CAPACIDADE_MAX_CARTEIRA) / 1000
    for f in fatores_vol
]

# Cores: verde = auto-financiado (funding == 0), vermelho = precisa de aporte
cores_fund = [COR_MARGEM if v < 0.001 else COR_ALERTA for v in funding_necessario]
bars = axes[1].bar(rotulos_vol, funding_necessario, color=cores_fund, alpha=0.85)

# Se todos forem zero, exibir mensagem de auto-financiamento no painel
if all(v < 0.001 for v in funding_necessario):
    axes[1].set_ylim(0, 1)
    axes[1].text(0.5, 0.5, "Todos os cenários são auto-financiados\ncom a capacidade máxima estimada",
                 ha="center", va="center", transform=axes[1].transAxes,
                 fontsize=11, color=COR_MARGEM, fontweight="bold")
else:
    for bar, val in zip(bars, funding_necessario):
        if val > 0.001:
            axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                         f"R${val:,.0f}K", ha="center", va="bottom", fontsize=9)
        else:
            axes[1].text(bar.get_x() + bar.get_width()/2,
                         axes[1].get_ylim()[1] * 0.05,
                         "Auto-fin.", ha="center", va="bottom", fontsize=8, color=COR_MARGEM)

axes[1].set_title("Necessidade de Funding Adicional por Cenário de Volume (R$ mil)", fontsize=10)
axes[1].set_xlabel("Fator de Volume")
axes[1].set_ylabel("Funding Necessário (R$ mil)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R${x:,.0f}K"))

plt.tight_layout()
salvar(fig, "04_ciclo_caixa")
plt.show()

print("\n── Ciclo de Caixa ──────────────────────")
print(f"  Spread líquido base mensal : R$ {spread_liq_mensal_base:,.0f}")
for f, fund in zip(fatores_vol, funding_necessario):
    status = "Auto-financiado ✅" if fund < 0.001 else f"Funding externo: R$ {fund:,.0f}K ⚠️"
    print(f"  Volume ×{f:.1f}               : {status}")


---

## Análise 3 — O modelo financeiro escala junto com o varejo ou há um teto de crescimento que o comprador herdaria?

> *"Utilizando o framework da Teoria das Restrições, esta análise identifica o gargalo financeiro da empresa-alvo. Ao traçar a curva de demanda por capital contra a oferta disponível, isolamos o "Ponto de Ruptura". Este é o fator exato de volume onde o modelo de crédito atual deixa de ser um facilitador de vendas e passa a ser uma restrição ao crescimento."*

**Framework:** Teoria das Restrições — identificação do gargalo financeiro  
**Entrega:** Curva de capital necessário versus capital disponível com ponto de ruptura

**Como este script responde à pergunta:**
> O script traça a curva de capital necessário como função do volume e a confronta com a capacidade máxima. Um único painel responde à pergunta:
> 1. **Capital necessário vs. capacidade máxima (curva + linha):** A curva azul sobe linearmente com o volume — quanto mais pedidos, mais capital precisa estar imobilizado na carteira. A linha tracejada vermelha é o teto fixo de capacidade. O ponto onde a curva cruza a linha é o teto de crescimento: o fator de volume a partir do qual a operação de crédito próprio precisa de capital adicional para sustentar o varejo. A zona sombreada em vermelho acima do cruzamento é a área de funding externo obrigatório — o teto real de crescimento do modelo atual, que o comprador precisaria endereçar para crescer além desse ponto.

**Análise do Resultado:**
O cruzamento da curva de capital necessário com a linha de capacidade máxima define objetivamente o teto de crescimento do modelo atual. A zona sombreada em vermelho acima desse ponto é a área de funding externo obrigatório — o comprador que pretender crescer além desse limite precisará endereçar a estrutura de capital antes de executar qualquer plano de expansão. Se o ponto de ruptura está próximo do volume atual, a tese de aquisição baseada em escala rápida requer revisão imediata; se está distante, o modelo tem fôlego para crescer com a estrutura existente por um período relevante.

In [ ]:
# ─── Teoria das Restrições: ponto de ruptura do capital ───────────────────────
volumes = np.linspace(0.5, 4.0, 50)
capital_necessario_curve = capital_mensal * volumes
capital_disponivel_line  = [CAPACIDADE_MAX_CARTEIRA] * len(volumes)

ponto_ruptura_vol = CAPACIDADE_MAX_CARTEIRA / capital_mensal if capital_mensal > 0 else None

fig, ax = plt.subplots(figsize=(14, 7))
fig.suptitle("Bloco 4 — Teto de Crescimento: Capital Necessário vs. Disponível",
             fontsize=13, fontweight="bold")

ax.plot(volumes, [c / 1000 for c in capital_necessario_curve],
        color=COR_RECEITA, linewidth=2.5, label="Capital Necessário na Carteira")
ax.axhline(CAPACIDADE_MAX_CARTEIRA / 1000, color=COR_ALERTA, linestyle="--",
           linewidth=2, label=f"Capacidade Máxima Estimada (R${CAPACIDADE_MAX_CARTEIRA/1000:.0f}K)")

if ponto_ruptura_vol:
    ax.axvline(ponto_ruptura_vol, color=COR_ROXO, linestyle=":", linewidth=1.5,
               label=f"Teto de crescimento: ×{ponto_ruptura_vol:.1f} volume atual")
    ax.fill_between(volumes,
                    [c / 1000 for c in capital_necessario_curve],
                    CAPACIDADE_MAX_CARTEIRA / 1000,
                    where=[c > CAPACIDADE_MAX_CARTEIRA for c in capital_necessario_curve],
                    alpha=0.15, color=COR_ALERTA, label="Zona de funding externo")

ax.set_xlabel("Fator de Volume (×base atual)", fontsize=10)
ax.set_ylabel("Capital (R$ mil)", fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R${x:,.0f}K"))
ax.legend(fontsize=9)
ax.set_title("")

plt.tight_layout()
salvar(fig, "04_teto_crescimento")
plt.show()

if ponto_ruptura_vol:
    print(f"\n── Teto de Crescimento ──────────────────────")
    print(f"  Capital mensal base       : R$ {capital_mensal:,.0f}")
    print(f"  Capacidade máxima         : R$ {CAPACIDADE_MAX_CARTEIRA:,.0f}")
    print(f"  Teto de crescimento       : ×{ponto_ruptura_vol:.1f} o volume atual")
    print(f"  Receita no teto           : R$ {receita_mensal * ponto_ruptura_vol:,.0f}/mês")
    print(f"  Acima do teto             : funding externo necessário")


---

## Análise 4 — Quais são os vetores de ruptura prioritários e quais os gatilhos que o comprador precisará monitorar?

> *"Riscos financeiros não são apenas números; são eventos operacionais. A Matriz GUT (Gravidade, Urgência e Tendência) prioriza os vetores de ruptura, separando o ruído estatístico das ameaças reais. Para cada risco, define-se um Gatilho Mensurável (KPI de alerta) e uma ação de mitigação, transformando a análise de risco em um plano de ação para os primeiros 100 dias pós-fechamento."*

**Framework:** Matriz GUT — priorização de riscos estruturais  
**Entrega:** Ranking de vetores de ruptura com score GUT e gatilhos mensuráveis

**Como este script responde à pergunta:**
> O script aplica a Matriz GUT a cinco vetores de risco estrutural, calcula o score automaticamente e prioriza por criticidade. Um único painel responde à pergunta:
> 1. **Barras horizontais de score GUT por risco:** Ordenadas do maior para o menor score, coloridas em vermelho (GUT ≥ 40), laranja (GUT ≥ 20) e verde (abaixo de 20). Para cada barra, o texto lateral exibe o score e o gatilho mensurável — a condição específica que, se atingida, aciona aquele risco. A barra mais longa é o vetor de ruptura prioritário: o risco que mais provavelmente se materializa primeiro sem ação preventiva — informação direta para o plano de 100 dias do comprador.

**Análise do Resultado:**
O ranking de scores GUT transforma a análise de risco em uma lista de prioridades de ação para os primeiros 100 dias pós-fechamento. Os vetores com score mais alto — barras vermelhas — são os que combinam maior gravidade potencial, maior urgência de resposta e tendência de piora: as vulnerabilidades que o comprador herda no dia um e que precisam de plano de mitigação antes do fechamento. Cada barra exibe o gatilho mensurável correspondente — a condição específica que, se atingida, aciona aquele vetor de risco. Esses gatilhos devem ser incorporados diretamente ao painel de monitoramento da nova gestão, permitindo resposta automática antes que o risco se materialize em impacto financeiro.

In [ ]:
# ─── Matriz GUT dos vetores de ruptura ────────────────────────────────────────
# G = Gravidade (1-5) | U = Urgência (1-5) | T = Tendência (1-5)
riscos_gut = [
    {
        "risco"     : "Concentração de inadimplência",
        "descricao" : "Spike de cancelamentos/anomalias em curto período",
        "G": 5, "U": 3, "T": 3,
        "gatilho"   : f"Taxa de anomalia > {TAXA_INADIMPLENCIA*200:.1f}% (2× base)",
        "mitigacao" : "Reserva de capital de 3× o spread líquido mensal",
    },
    {
        "risco"     : "Falta de liquidez por crescimento acelerado",
        "descricao" : "Volume cresce mais rápido do que o caixa disponível suporta",
        "G": 5, "U": 4, "T": 4,
        "gatilho"   : f"Volume acima de ×{ponto_ruptura_vol:.1f} sem funding adicional",
        "mitigacao" : "Monitoramento mensal da razão capital/carteira",
    },
    {
        "risco"     : "Deterioração de margem operacional",
        "descricao" : "Custo operacional cresce sem ganho proporcional de spread",
        "G": 4, "U": 2, "T": 3,
        "gatilho"   : "Spread líquido mensal cai abaixo de R$ 5.000",
        "mitigacao" : "Revisão trimestral das premissas de custo",
    },
    {
        "risco"     : "Concentração regional de risco",
        "descricao" : "Shock econômico regional amplifica inadimplência em SP/RJ/MG",
        "G": 4, "U": 2, "T": 2,
        "gatilho"   : "Anomalia nas top 3 regiões > 2× a média nacional",
        "mitigacao" : "Limites de exposição por região na carteira",
    },
    {
        "risco"     : "Mudança no ambiente regulatório",
        "descricao" : "Regulação de pagamentos ou crédito altera as condições da operação",
        "G": 5, "U": 1, "T": 1,
        "gatilho"   : "Publicação de regulamentação específica para fintechs varejistas",
        "mitigacao" : "Monitoramento regulatório contínuo; due diligence jurídica pré-aquisição",
    },
]

df_gut = pd.DataFrame(riscos_gut)
df_gut["GUT"] = df_gut["G"] * df_gut["U"] * df_gut["T"]
df_gut = df_gut.sort_values("GUT", ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
fig.suptitle("Bloco 4 — Matriz GUT: Priorização dos Vetores de Ruptura",
             fontsize=13, fontweight="bold")

cores_gut = [COR_ALERTA if g >= 40 else COR_DESTAQUE if g >= 20 else COR_MARGEM
             for g in df_gut["GUT"]]
bars = ax.barh(df_gut["risco"], df_gut["GUT"], color=cores_gut, alpha=0.85)
ax.set_xlabel("Score GUT (G × U × T)")
ax.set_title("")
for bar, val, gat in zip(bars, df_gut["GUT"], df_gut["gatilho"]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f"Score {val} | Gatilho: {gat[:50]}...",
            va="center", fontsize=7.5, color="#333333")

plt.tight_layout()
salvar(fig, "04_matriz_gut")
plt.show()

print("\n── Matriz GUT — Ranking de Riscos ──────────────────────")
for _, row in df_gut.iterrows():
    nivel = "🔴" if row["GUT"] >= 40 else "⚠️ " if row["GUT"] >= 20 else "✅"
    print(f"  {nivel} [{row['GUT']:>3}] {row['risco']:<40} | Gatilho: {row['gatilho']}")
    print(f"          Mitigação: {row['mitigacao']}")
    print()


In [ ]:
# ─── Score do Bloco 4 ────────────────────────────────────────────────────────
_b4_capital_ok    = capital_mensal < CAPACIDADE_MAX_CARTEIRA * 0.7
_b4_teto_ok       = ponto_ruptura_vol is None or ponto_ruptura_vol >= 2.0
_b4_gut_critico   = df_gut[df_gut["GUT"] >= 50].shape[0] > 0

if not _b4_capital_ok or _b4_gut_critico:
    _b4_score = 1
    _b4_sinal = "🔴 Capital já próximo do limite — crescimento adicional exige funding externo"
    _b4_cor   = COR_ALERTA
elif not _b4_teto_ok:
    _b4_score = 2
    _b4_sinal = "⚠️  Teto de crescimento identificado — modelo escala com restrição de capital"
    _b4_cor   = COR_DESTAQUE
else:
    _b4_score = 3
    _b4_sinal = "✅ Estrutura de capital compatível com o crescimento projetado no horizonte analisado"
    _b4_cor   = COR_MARGEM

_b4_cond = (
    f"Definir política de capital máximo da carteira (sugerido: R$ {CAPACIDADE_MAX_CARTEIRA:,.0f}) "
    f"e plano de funding antes de qualquer expansão de volume acima de ×{ponto_ruptura_vol:.1f}."
    if _b4_score <= 2
    else "Estrutura de capital compatível com a tese de aquisição."
)

print("=" * 60)
print("SÍNTESE — BLOCO 4: CAPITAL E ESCALABILIDADE")
print("=" * 60)
print(f"  Capital em aberto médio mensal : R$ {capital_mensal:,.0f}")
print(f"  Capacidade máxima estimada     : R$ {CAPACIDADE_MAX_CARTEIRA:,.0f}")
if ponto_ruptura_vol:
    print(f"  Teto de crescimento            : ×{ponto_ruptura_vol:.1f} volume atual")
print(f"  Riscos GUT críticos (≥50)      : {df_gut[df_gut['GUT']>=50].shape[0]}")
print(f"  Risco prioritário              : {df_gut.iloc[0]['risco']} (GUT={df_gut.iloc[0]['GUT']})")
print()
print(f"  Score do Bloco: {_b4_score}/3")
print(f"  Sinal        : {_b4_sinal}")
print()
print(f"  Condicionante: {_b4_cond}")
print("=" * 60)


---
*Próximo notebook: `05_cenarios_recomendacao_financeira.ipynb` — Comprar, não comprar ou condicionar — e qual é o custo financeiro de errar em cada direção?*

> Esta análise faz parte do **Projeto Fictus**, conduzido pela Lufi Data Consulting. Os três módulos analíticos — Vendas, Logística e Finanças — compõem a base do Relatório de Recomendação de Aquisição.
